# Text-to-SQL Agent Tool with AST Read-Only Validation

**Reference:** Slide 33 (*Building Agentic Workflows with RAG on AWS Bedrock*)

### Why Defense-in-Depth AST Validation is Mandatory
Allowing autonomous agents or LLMs to execute generated SQL queries against enterprise databases without strict validation creates critical vulnerabilities:
1. **Accidental / Malicious Data Deletion**: Prompts or hallucinations generating `DELETE`, `UPDATE`, or `DROP TABLE`.
2. **SQL Injection Chaining**: Semicolon-delimited queries (`SELECT ...; DROP TABLE ...`).
3. **Comment-based Evasions**: Obfuscating destructive statements inside SQL comments.

In this demo, we implement the complete **4-Step Text-to-SQL Agent Tool**:
1. **Schema Inspection**: Introspecting database schema into structured LLM context.
2. **SQL Generation**: Generating SQL via Bedrock Converse API.
3. **AST Read-Only Validation**: Deterministically rejecting any non-SELECT statement, comment, or forbidden keyword.
4. **Engine-Level Authorizer Execution**: SQLite / database engine-level authorizer restricting execution exclusively to read-only actions.

In [ ]:
from text_to_sql_agent import seed_sample_database, TextToSQLPipeline, SecurityValidationError

conn = seed_sample_database()
pipeline = TextToSQLPipeline(connection=conn, mock_mode=True)
print("Seeded in-memory retail database and initialized TextToSQLPipeline.")

## 1. Schema Inspection
Introspects tables, columns, and data types to construct the system prompt context.

In [ ]:
schema_context = pipeline.inspector.format_schema_prompt()
print(schema_context)

## 2. Safe Query Execution (Read-Only SELECT)
Translates natural language to SQL, validates AST, and executes.

In [ ]:
question = "Show top customers ranked by total revenue spent"
response = pipeline.run(question)

print(f"Question: {response['question']}")
print(f"Generated & Validated SQL: {response['validated_sql']}")
print(f"Execution Time: {response['result']['execution_time_ms']} ms")
print("\nTabular Result:")
for row in response['result']['rows']:
    print(" ", row)

## 3. Security Hardening: Blocking Malicious Queries & Injections
We test various malicious attacks against the AST validator.

In [ ]:
attacks = [
    "DROP TABLE customers;",
    "SELECT * FROM customers; DELETE FROM orders;",
    "UPDATE products SET unit_price = 0.00;",
    "INSERT INTO customers VALUES (10, 'Attacker', 'a@a.com', 'x');",
    "SELECT * FROM customers; -- DROP TABLE orders;"
]

for atk in attacks:
    try:
        pipeline.validator.validate(atk)
        print(f"[UNEXPECTED ALLOW]: {atk}")
    except SecurityValidationError as e:
        print(f"[BLOCKED SAFELY]: '{atk}' --> Reason: {e}")

## 4. Key Security Takeaways
- Never rely solely on system prompt instructions like 'Only generate SELECT queries'. LLMs can be tricked via prompt injection.
- Use deterministic code-level AST parsing combined with engine-level authorizer permissions.